# Notebook 01 — Why Single-Cell? Technology and Data Shape

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 01 of 12  
**Time estimate:** 60 minutes

> Bulk RNA-seq tells you the average expression of thousands of cells mixed together.
> Single-cell RNA-seq tells you the expression of each cell individually.
> The data shape changes from a vector to a matrix — and that changes everything.

---
## Step 1 — Motivation

A tumour biopsy contains cancer cells, immune cells, fibroblasts, endothelial cells.
Bulk RNA-seq averages over all of them. scRNA-seq resolves individual cells and their
states — enabling cell type identification, rare cell detection, and understanding of
cellular heterogeneity that is invisible to bulk methods.

---
## Step 2 — Intuition

**Bulk RNA-seq:** one tube of cells → one library → one row of expression values.
Output: a vector of length G (number of genes).

**scRNA-seq:** each cell barcoded individually → many libraries → many rows of
expression values. Output: a matrix of shape (cells × genes).

**UMI counts, not reads:** Each RNA molecule is tagged with a Unique Molecular
Identifier (UMI) before PCR amplification. Counting distinct UMIs rather than total
reads removes PCR duplicate bias — this is why scRNA-seq counts are intrinsically
better calibrated than bulk read counts.

**The sparsity problem:** A typical cell expresses 2,000–5,000 genes, but the genome
has 20,000+ genes. Most entries in the cell × gene matrix are zero — the matrix is
~90–95% sparse. Many zeros are real (gene not expressed), but some are **dropout**
(gene expressed but no mRNA captured).

---
## Step 3 — Biological Background

**10x Genomics droplet microfluidics (Chromium):**
1. Cells and gel beads (with barcodes + oligo-dT) are co-encapsulated in nanoliter
   oil droplets (GEMs — gel bead in emulsion)
2. Each cell is paired with one gel bead — the bead barcode identifies the cell
3. mRNA from that cell is captured, reverse-transcribed, and tagged with:
   - **Cell barcode (16 nt):** which cell this came from
   - **UMI (12 nt):** which individual mRNA molecule
4. Libraries from all droplets are pooled and sequenced on Illumina
5. Reads are demultiplexed by cell barcode → UMI counted per gene per cell

**Key technical artefacts to know:**
- **Doublets:** Two cells captured in one droplet → one barcode, two cells' RNA
- **Ambient RNA:** mRNA from lysed cells contaminates the solution → background signal
- **Dropout:** Stochastic failure to capture low-abundance mRNA → false zeros

**Typical numbers (10x Genomics v3):**
- ~2000–10,000 cells per run
- ~1000–5000 genes detected per cell
- ~500–3000 UMIs per cell (median)

---
## Step 4 — Mathematical Explanation

**The count matrix:**  
Let $X \in \mathbb{Z}_{\geq 0}^{n \times p}$ where $n$ = cells, $p$ = genes.  
$X_{ij}$ = number of distinct UMIs for gene $j$ in cell $i$.

**Library size (total UMI count per cell):**  
$$s_i = \sum_{j=1}^{p} X_{ij}$$

Cells with very low $s_i$ are likely dead/damaged cells (empty droplets or poor
capture). Cells with very high $s_i$ may be doublets.

**Mitochondrial fraction:**  
$$\text{pct\_mito}_i = \frac{\sum_{j \in \text{mito}} X_{ij}}{s_i}$$

Dying cells have compromised cytoplasmic membranes, losing cytoplasmic RNA but
retaining mitochondrial RNA (mitochondria are membrane-enclosed). A high
mitochondrial fraction (>20–25%) indicates cell death or poor quality.

In [ ]:
# Step 6 — Python: Build a synthetic cell × gene count matrix
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

rng = np.random.default_rng(42)

# Simulate 200 cells, 1000 genes, 3 cell types
N_CELLS = 200
N_GENES = 1000
N_CELL_TYPES = 3
N_MITO_GENES = 13  # human has 13 mitochondrially encoded proteins

# Gene names
gene_names = [f'Gene{i:04d}' for i in range(N_GENES - N_MITO_GENES)]
gene_names += [f'MT-{g}' for g in ['ND1','ND2','COX1','COX2','ATP8','ATP6','COX3',
                                     'ND3','ND4L','ND4','ND5','ND6','CYB']]
cell_barcodes = [f'CELL_{i:04d}' for i in range(N_CELLS)]

# Assign cell types
cell_types = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS,
                         p=[0.4, 0.35, 0.25])

# Gene expression profiles per cell type
# Each cell type has a set of marker genes with higher expression
def make_count_matrix(n_cells, n_genes, cell_types, rng):
    X = np.zeros((n_cells, n_genes), dtype=np.int32)
    cell_type_markers = {
        'T_cell':   {'base': 200, 'marker_genes': list(range(0, 30)),   'marker_boost': 5},
        'B_cell':   {'base': 180, 'marker_genes': list(range(30, 60)),  'marker_boost': 6},
        'Monocyte': {'base': 250, 'marker_genes': list(range(60, 100)), 'marker_boost': 4},
    }
    for i, ct in enumerate(cell_types):
        profile = cell_type_markers[ct]
        base_expr = rng.exponential(0.5, n_genes)
        # Boost marker genes
        base_expr[profile['marker_genes']] *= profile['marker_boost']
        # Boost mitochondrial genes slightly for some cells (simulate stress)
        if rng.random() < 0.05:  # 5% "stressed" cells
            base_expr[-N_MITO_GENES:] *= 6
        # Scale by library size
        lib_size = int(rng.normal(2000, 400))
        lib_size = max(300, lib_size)
        probs = base_expr / base_expr.sum()
        X[i, :] = rng.multinomial(lib_size, probs)
    return X

X = make_count_matrix(N_CELLS, N_GENES, cell_types, rng)

# Basic stats
library_sizes = X.sum(axis=1)
n_genes_per_cell = (X > 0).sum(axis=1)
mito_counts = X[:, -N_MITO_GENES:].sum(axis=1)
pct_mito = mito_counts / library_sizes * 100
sparsity = (X == 0).mean()

print(f'Count matrix shape: {X.shape} (cells × genes)')
print(f'Matrix sparsity: {sparsity*100:.1f}% zeros')
print(f'Median library size: {np.median(library_sizes):.0f} UMIs/cell')
print(f'Median genes per cell: {np.median(n_genes_per_cell):.0f}')
print(f'Median mitochondrial %: {np.median(pct_mito):.1f}%')
print(f'Cells with >20% mito: {(pct_mito > 20).sum()}')

In [ ]:
# Step 7 — Visualization: Technology comparison and data shape
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 3, fig)

# Panel A: Bulk vs single-cell data shape
ax = fig.add_subplot(gs[0, :2])
ax.axis('off')
# Bulk: single row
bulk_data = rng.exponential(1, 8)
for j in range(8):
    h = min(bulk_data[j] / bulk_data.max() * 0.8, 0.8)
    ax.add_patch(plt.Rectangle((j * 0.07 + 0.05, 0.65), 0.06, h,
                                 facecolor='steelblue', alpha=0.8, edgecolor='white'))
ax.text(0.5, 0.62, 'Bulk RNA-seq: 1 row (average of all cells)', ha='center', fontsize=9)
ax.text(0.5, 0.56, 'Output: vector of length G', ha='center', fontsize=8, style='italic')
# Single-cell: multiple rows
for i in range(5):
    row_data = rng.exponential(0.8, 8)
    for j in range(8):
        h = min(row_data[j] / row_data.max() * 0.07, 0.07)
        ax.add_patch(plt.Rectangle((j * 0.07 + 0.05, 0.38 - i * 0.09), 0.06, h,
                                     facecolor='tomato', alpha=0.7, edgecolor='white'))
ax.text(0.5, 0.10, 'scRNA-seq: N rows (one per cell)', ha='center', fontsize=9)
ax.text(0.5, 0.03, 'Output: matrix of shape (cells × genes)', ha='center', fontsize=8, style='italic')
ax.set_title('A. Bulk vs. single-cell transcriptomics data shape', fontsize=10, fontweight='bold')
ax.set_xlim(0, 0.7)
ax.set_ylim(0, 1.1)

# Panel B: Library size distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(library_sizes, bins=30, color='steelblue', alpha=0.8, edgecolor='none')
ax2.set_xlabel('Total UMI count (library size)')
ax2.set_ylabel('Number of cells')
ax2.set_title('B. Library size distribution\n(filter cells below ~300)')
ax2.axvline(300, color='red', ls='--', label='min threshold')
ax2.legend(fontsize=8)

# Panel C: Genes per cell
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(n_genes_per_cell, bins=30, color='forestgreen', alpha=0.8)
ax3.set_xlabel('Genes detected per cell')
ax3.set_ylabel('Number of cells')
ax3.set_title('C. Genes detected per cell\n(low n_genes → poor quality)')

# Panel D: Mitochondrial fraction
ax4 = fig.add_subplot(gs[1, 1])
colors4 = ['tomato' if p > 20 else 'steelblue' for p in pct_mito]
ax4.scatter(library_sizes, pct_mito, c=colors4, alpha=0.5, s=10)
ax4.axhline(20, color='red', ls='--', label='20% mito threshold')
ax4.set_xlabel('Library size (UMIs)')
ax4.set_ylabel('% mitochondrial UMIs')
ax4.set_title('D. Mitochondrial fraction vs. library size\n(high mito = dying cells)')
ax4.legend(fontsize=8)

# Panel E: Sparsity illustration
ax5 = fig.add_subplot(gs[1, 2])
show = X[:40, :60].copy().astype(float)
show[show > 0] = 1
ax5.imshow(show, aspect='auto', cmap='Blues', interpolation='none')
ax5.set_xlabel('Gene index')
ax5.set_ylabel('Cell index')
ax5.set_title(f'E. Count matrix sparsity\n({sparsity*100:.0f}% zeros, {(1-sparsity)*100:.0f}% non-zero)')

plt.suptitle('Module 10: Single-Cell RNA-seq — Technology and Data Shape', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('singlecell_intro.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A cell has 50 total UMI counts. Should you include it in analysis? What are the
   two possible explanations for such a low count?
2. A cell has 8000 UMI counts when the median is 2000. What is the most likely
   artefact? What downstream effects would this cell have if not removed?
3. Compute the sparsity of your simulated matrix. What fraction of cells have zero
   counts for 'Gene0010'?
4. What is the difference between a UMI and a read? Why does counting UMIs rather
   than reads give more accurate expression estimates?

---
## Step 10 — Quiz

1. What is the output data shape of a scRNA-seq experiment?
2. What is a UMI and why is it used?
3. What are the three main artefacts in 10x Genomics data?
4. What does a high mitochondrial fraction indicate about a cell?
5. What is dropout in the context of scRNA-seq?

---
## Step 12 — Reflection

> *[In one paragraph: explain to someone who only knows bulk RNA-seq why the
> transition to single-cell data requires new computational methods at every step
> of the analysis pipeline.]*

---
*Next: `02_scrna_qc_and_cell_filtering.ipynb`*